# Lab 06: Patrones de Uso de Modelos Locales

## 🎯 Objetivos

Aprenderás a instalar, configurar y usar modelos de lenguaje locales con **5 patrones diferentes**:

1. **Ollama** (más fácil - todo automático)
2. **llama-cpp-python** (Python puro, sin servidor)
3. **llama.cpp** (compilado con servidor HTTP)
4. **vLLM** (producción GPU - PagedAttention)
5. **TGI** (producción GPU - FlashAttention, de HuggingFace)

Para cada patrón, aprenderás **3 métodos de uso** (de alto a bajo nivel):

1. **LangChain** → Framework alto nivel (fácil)
2. **requests** → Librería HTTP de Python (medio)
3. **curl** → Línea de comandos (bajo nivel)

## 📚 Prerequisitos

- Haber leído `lab_05_ecosistema_llm_local.md`
- Python 3.10+
- Este notebook funciona en **Google Colab** (con GPU T4) y localmente

## ✅ Características del Notebook

- ✅ **Autocompleto:** Crea directorios y descarga automáticamente
- ✅ **Compatible Colab:** Funciona en Google Colab sin modificaciones
- ✅ **GPU Ready:** Instrucciones para CPU y GPU NVIDIA
- ✅ **Explicaciones progresivas:** Conceptos explicados cuando aparecen
- ✅ **Ejecutable de principio a fin:** Solo "Run All"

---
# PARTE 0: Setup Inicial y Conceptos

## 🔧 Configuración del Entorno

In [ ]:
# Detectar entorno (Colab vs Local)
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

# Crear directorios necesarios
os.makedirs('./models', exist_ok=True)
os.makedirs('./logs', exist_ok=True)

print(f"🌍 Entorno detectado: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"📂 Directorio de trabajo: {os.getcwd()}")
print(f"✅ Directorios creados: ./models/, ./logs/")

🌍 Entorno detectado: Google Colab
📂 Directorio de trabajo: /content
✅ Directorios creados: ./models/, ./logs/


In [ ]:
# Instalar dependencias base (necesarias para todo)
!pip install -q requests tqdm

## 💡 Conceptos Clave

### Glosario de Términos Técnicos

Antes de empezar, definimos términos que usaremos constantemente:

| Término | Definición | Ejemplo |
|---------|------------|---------|
| **Daemon** | Proceso que corre en **background** (segundo plano), siempre activo esperando peticiones. No tiene interfaz visual. | `ollama serve` crea un daemon que escucha en el puerto 11434 |
| **SDK** | **Software Development Kit** - Conjunto de herramientas y librerías para desarrollar aplicaciones. | `langchain-ollama` es un SDK para usar Ollama desde Python |
| **API** | **Application Programming Interface** - Forma estándar de comunicarse con un servicio. | La API de Ollama permite enviar prompts y recibir respuestas |
| **Servidor HTTP** | Programa que escucha peticiones HTTP y responde. | Ollama corre un servidor HTTP en `localhost:11434` |
| **Cargar en memoria** | Leer el modelo desde disco y ponerlo en RAM/VRAM para usarlo. | llama-cpp-python carga el modelo cada vez que ejecutas el script |
| **Proceso** | Instancia de un programa en ejecución. | Cuando ejecutas `python script.py`, creas un proceso |

### ¿Cuándo se activa un servidor y cómo?

**Hay 2 formas de iniciar un servidor:**

| Método | Cuándo usarlo | Ejemplo |
|--------|---------------|---------|
| **Desde terminal** | En producción, más control | `ollama serve` o `./llama-server -m modelo.gguf` |
| **Desde Python** | En notebooks, automatización | `subprocess.Popen(['ollama', 'serve'])` |

**Buena práctica:**
- **Desarrollo/Notebooks** → Desde Python (este notebook)
- **Producción** → Desde terminal o como servicio del sistema (systemd)

### ¿Memoria vs HTTP? ¿Qué es más eficiente?

| Método | Latencia | Cuándo modelo está en memoria |
|--------|----------|-------------------------------|
| **Carga directa (llama-cpp-python)** | ⚡ ~0ms overhead | Solo mientras tu script corre |
| **HTTP (Ollama, vLLM, llama-server)** | ~1-5ms overhead | **Mientras el servidor esté corriendo** |

**Respuesta a tu pregunta:**
> "¿Cuando es con conexión HTTP continúa el modelo cargado en memoria hasta tanto no se apague el servidor?"

**Sí, exactamente.** Cuando inicias un servidor (Ollama, llama-server, vLLM), el modelo se carga en memoria (RAM o VRAM) y **permanece cargado** hasta que detienes el servidor. Esto tiene ventajas:

- ✅ Primera petición: lenta (carga el modelo)
- ✅ Peticiones siguientes: rápidas (modelo ya cargado)
- ✅ Múltiples clientes pueden usar el mismo modelo cargado

### ¿Qué son estos "niveles"?

```
Alto nivel     → LangChain    → Abstracto, 2-3 líneas de código
               ↓
Nivel medio    → requests     → Control medio, 5-7 líneas
               ↓
Bajo nivel     → curl         → Control total, terminal
```

### ¿Por qué aprender los 3 métodos?

- **LangChain**: Para desarrollo rápido (agentes, RAG, chains)
- **requests**: Para entender qué hace LangChain por debajo
- **curl**: Para debugging, testing y portabilidad

### Los 4 Patrones

| Patrón | Servidor HTTP | Descarga Modelos | Dificultad | Mejor Para |
|--------|--------------|------------------|------------|------------|
| **Ollama** | ✅ Automático | Automática | ⭐ | Aprendizaje |
| **llama-cpp-python** | ❌ No | Manual | ⭐⭐ | Scripts únicos |
| **llama.cpp** | ✅ Manual | Manual | ⭐⭐⭐ | Control total |
| **vLLM** | ✅ Manual | Automática | ⭐⭐ | Producción |

### ¿Dónde vive el modelo? (Memoria vs Disco)

| Patrón | En disco | En memoria (RAM/VRAM) |
|--------|----------|----------------------|
| **Ollama** | `~/.ollama/models/` | Mientras servidor corre |
| **llama-cpp-python** | Donde lo descargaste (`./models/`) | Solo durante ejecución del script |
| **llama.cpp** | Donde lo descargaste | Mientras servidor corre |
| **vLLM** | `~/.cache/huggingface/hub/` | Mientras servidor corre |

### Conceptos HTTP/API que aprenderás

- **HTTP**: Protocolo de comunicación web
- **GET**: Obtener información
- **POST**: Enviar datos
- **JSON**: Formato de datos (como diccionarios Python)
- **Endpoint**: URL específica que hace algo
- **localhost**: Tu propia computadora
- **Puerto**: Número que identifica un servicio (ej: 11434)

---
# PARTE 1: Patrón Ollama (El Más Fácil)

## 🦙 ¿Por qué empezar con Ollama?

- Instalación en 1 comando
- Descarga de modelos automática
- Servidor se gestiona automáticamente
- Compatible con API de OpenAI

**Estructura:**
```
Tu código → HTTP → Servidor Ollama → Motor llama.cpp → Modelo
```

## 1.1 Instalación de Ollama

In [ ]:
%%bash
# Instalar zstd, requerido por Ollama
apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Descargar e instalar Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Verificar instalación
which ollama
ollama --version

/usr/local/bin/ollama


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Iniciar servidor Ollama en background
# En Colab/Linux necesitamos iniciarlo manualmente
import subprocess
import time

# Iniciar servidor en background
print("🚀 Iniciando servidor Ollama...")
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Esperar a que inicie
time.sleep(5)

# Verificar que está corriendo
import requests
try:
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    if response.status_code == 200:
        print("✅ Servidor Ollama está corriendo en localhost:11434")
    else:
        print(f"⚠️ Servidor responde pero con código: {response.status_code}")
except Exception as e:
    print(f"❌ Error al conectar: {e}")
    print("Intenta ejecutar esta celda nuevamente")

🚀 Iniciando servidor Ollama...
✅ Servidor Ollama está corriendo en localhost:11434


**¿Qué hicimos?**

1. Descargamos Ollama desde https://ollama.com
2. Lo instalamos en `/usr/local/bin/`
3. Iniciamos el servidor en el puerto **11434**

**¿Dónde se guardan los modelos?**
- Linux/Colab: `~/.ollama/models/`
- macOS: `~/.ollama/models/`
- Windows: `C:\Users\<usuario>\.ollama\models\`

## 1.2 Descargar Modelo

In [ ]:
%%bash
# Descargar modelo pequeño para pruebas (~2GB)
# Esto puede tomar 2-5 minutos dependiendo de tu conexión
ollama pull llama3.2

# Ver modelos descargados
echo ""
echo "📦 Modelos instalados:"
ollama list


📦 Modelos instalados:
NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 5.0 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   4% ▕                  ▏  83 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   6% ▕█                 ▏ 120 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   7% ▕█                 ▏ 147 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   9% ▕█                 ▏ 182 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  10% ▕█                 ▏ 197 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  11% ▕██                ▏ 225 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  13% ▕██                ▏ 257 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  14% ▕██     

**¿Qué acabamos de hacer?**

- Descargamos `llama3.2` (3B parámetros)
- Ollama lo guardó en formato GGUF cuantizado (Q4)
- El modelo está listo para usar

**Otros modelos disponibles:**
- `tinyllama` → 1.1B (muy pequeño, rápido)
- `llama3.2` → 3B (balance)
- `llama3.1` → 8B (mejor calidad)
- `mistral` → 7B (excelente calidad)

## 1.3 Método 1: LangChain (Alto Nivel)

### ¿Qué es LangChain?

Un framework de Python que **simplifica el uso de LLMs**.

**Ventajas:**
- Código muy corto (2-3 líneas)
- Mismo código funciona con diferentes proveedores (Ollama, OpenAI, etc.)
- Funciones avanzadas: agentes, RAG, memory, chains

In [ ]:
# Instalar LangChain para Ollama
!pip install -q langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama

# 1. Crear cliente
llm = ChatOllama(
    model="llama3.2",
    temperature=0.8  # Creatividad (0=determinista, 1=creativo)
)

# 2. Hacer pregunta
response = llm.invoke("What are the most important deep learning methods for fisheye image processing?")

# 3. Mostrar respuesta
print("💬 Respuesta:")
print(response.content)

💬 Respuesta:
For fisheye image processing, several deep learning methods have been proposed and evaluated. Here are some of the most important ones:

1. **Fisheye Camera Model Prediction**: This method involves predicting the fisheye camera model parameters (e.g., radial distortion, tangential distortion) using deep neural networks. These predictions can be used to correct distortions in images.

2. **Optical Flow Estimation for Fisheye Cameras**: Optical flow estimation is crucial for fisheye cameras, as it helps estimate the motion of points between two frames. Deep learning-based methods like DeepSimpleStrom (DSS) and PyramidalResidualFlow (PRF) have been proposed to estimate optical flows efficiently.

3. **Stereo Matching for Fisheye Images**: Stereo matching is used to estimate depth maps from stereo images. For fisheye cameras, stereo matching needs to be adapted due to the radial distortion. Deep learning-based methods like DeepSimpleStrom (DSS) and PyramidalResidualFlow (PRF) 

**Explicación del código:**

```python
llm = ChatOllama(model="llama3.2", temperature=0.7)
```
- `ChatOllama()` → Se conecta automáticamente a `localhost:11434`
- `model` → Qué modelo usar (debe estar descargado)
- `temperature` → Controla creatividad (0-1)

```python
response = llm.invoke("pregunta")
```
- `invoke()` → Envía la pregunta y espera respuesta completa
- Retorna un objeto con la respuesta

```python
print(response.content)
```
- `.content` → Extrae solo el texto de la respuesta

**¿Qué hace LangChain por ti automáticamente?**
1. Se conecta al servidor Ollama
2. Formatea la petición correctamente
3. Maneja errores
4. Parsea la respuesta

## 1.4 Método 2: requests (Nivel Medio)

### ¿Qué es requests?

Una librería de Python para **hacer peticiones HTTP**.

### ¿Por qué aprender esto?

- Entiendes **qué hace LangChain por debajo**
- Control total sobre la petición HTTP
- Útil para debugging y casos personalizados

### Conceptos necesarios:

- **HTTP**: Protocolo para comunicación web
- **POST**: Tipo de petición HTTP para enviar datos
- **JSON**: Formato de datos (como diccionarios Python)
- **Endpoint**: URL específica que hace algo (ej: `/api/generate`)

In [ ]:
import requests

# 1. Definir el endpoint (URL del servidor Ollama)
url = "http://localhost:11434/api/generate"

# 2. Preparar los datos a enviar (en formato de diccionario)
payload = {
    "model": "llama3.2",
    "prompt": "What are the most important deep learning methods for fisheye image processing?",
    "stream": False  # Queremos respuesta completa, no streaming
}

# 3. Hacer petición HTTP POST
response = requests.post(url, json=payload)

# 4. Convertir respuesta JSON a diccionario Python
result = response.json()

# 5. Extraer solo el texto generado
print("💬 Respuesta:")
print(result["response"])

💬 Respuesta:
Deep learning has revolutionized the field of fisheye image processing, enabling the development of robust and efficient algorithms for various applications. Here are some of the most important deep learning methods for fisheye image processing:

1. **Fisheye Camera Calibration using Deep Learning**: This method uses deep neural networks to estimate the camera intrinsic and extrinsic parameters from a single image or a set of images. Techniques such as 3D CNNs, Siamese networks, and convolutional neural networks (CNNs) have been proposed for this purpose.
2. **Fisheye Image Stitching using Deep Learning**: Fisheye image stitching is crucial for applications like virtual reality and augmented reality. Deep learning-based methods use CNNs and attention mechanisms to stitch fisheye images together, creating seamless and high-quality panoramas.
3. **Fisheye Camera Pose Estimation using Deep Learning**: This method uses deep neural networks to estimate the camera pose (position

**Explicación línea por línea:**

```python
url = "http://localhost:11434/api/generate"
```
Desglose de la URL:
- `http://` → Protocolo (cómo hablar con el servidor)
- `localhost` → Tu propia computadora (equivale a 127.0.0.1)
- `11434` → Puerto donde Ollama escucha peticiones
- `/api/generate` → Endpoint para generar texto

```python
payload = {"model": "llama3.2", "prompt": "...", "stream": False}
```
- `payload` → Los datos que enviamos al servidor
- `model` → Qué modelo queremos usar
- `prompt` → Tu pregunta o instrucción
- `stream` → `False` = queremos la respuesta completa de una vez

```python
response = requests.post(url, json=payload)
```
- `requests.post()` → Hace una petición HTTP POST
- `json=payload` → Convierte el diccionario a formato JSON automáticamente

```python
result = response.json()
```
- `.json()` → Convierte la respuesta JSON de vuelta a diccionario Python

```python
print(result["response"])
```
- `result` es un diccionario con varias claves
- La clave `"response"` contiene el texto generado

**Diferencia con LangChain:**
- **LangChain**: 3 líneas de código
- **requests**: 7 líneas de código
- **requests** da más control sobre el payload y la respuesta

In [ ]:
# Ver la respuesta completa de Ollama
import json

print("📊 Respuesta completa de Ollama:")
print(json.dumps(result, indent=2, ensure_ascii=False))

**Campos importantes en la respuesta:**

- `response` → El texto generado
- `model` → Modelo usado
- `total_duration` → Tiempo total (nanosegundos)
- `prompt_eval_count` → Tokens en tu pregunta
- `eval_count` → Tokens generados
- `done` → Si terminó la generación

## 1.5 Método 3: curl (Bajo Nivel)

### ¿Qué es curl?

Un **comando de terminal** para hacer peticiones HTTP.

### ¿Por qué aprender curl?

- Funciona en **cualquier lenguaje** (no solo Python)
- Útil para **probar APIs rápidamente**
- Esencial para **debugging**
- Fácil de **compartir** en documentación

### Conceptos nuevos:

- `%%bash` → Ejecuta comandos de terminal desde el notebook
- `-d` → Data (datos a enviar)
- `-s` → Silent (sin mostrar progreso)
- `|` → Pipe (pasa la salida a otro comando)

In [ ]:
%%bash
# Hacer petición con curl
curl -s http://localhost:11434/api/generate \
  -d '{
    "model": "llama3.2",
    "prompt": "¿Qué es Python en una oración?",
    "stream": false
  }' \
  | python3 -m json.tool

{
    "model": "llama3.2",
    "created_at": "2026-01-28T16:57:42.509761203Z",
    "response": "Python es un lenguaje de programaci\u00f3n interpretado y de alto nivel, ampliamente utilizado para desarrollar aplicaciones, scripts y herramientas en la industria de la tecnolog\u00eda.",
    "done": true,
    "done_reason": "stop",
    "context": [
        128006,
        9125,
        128007,
        271,
        38766,
        1303,
        33025,
        2696,
        25,
        6790,
        220,
        2366,
        18,
        271,
        128009,
        128006,
        882,
        128007,
        271,
        31282,
        66806,
        1560,
        13325,
        665,
        5203,
        477,
        5840,
        30,
        128009,
        128006,
        78191,
        128007,
        271,
        31380,
        1560,
        653,
        326,
        28102,
        11305,
        409,
        2068,
        5840,
        14532,
        2172,
        379,
        409,
 

**Desglose del comando:**

```bash
curl -s http://localhost:11434/api/generate
```
- `curl` → Comando para hacer peticiones HTTP
- `-s` → Silent (no mostrar barra de progreso)
- `http://...` → URL completa del endpoint

```bash
-d '{...}'
```
- `-d` → Data (enviar estos datos)
- `'{...}'` → JSON entre comillas simples
- Por defecto, `-d` usa método POST

```bash
\ (backslash al final de cada línea)
```
- Permite escribir el comando en múltiples líneas
- Hace el código más legible

```bash
| python3 -m json.tool
```
- `|` → Pipe: toma la salida de curl y la pasa a...
- `python3 -m json.tool` → Formatea el JSON bonito

**Equivalencia con requests:**

| curl | requests Python |
|------|----------------|
| `curl URL` | `requests.post(url, ...)` |
| `-d '{...}'` | `json=payload` |
| El JSON es idéntico | El JSON es idéntico |

**¿Cuándo usar curl?**
- Probar el servidor rápidamente
- Compartir ejemplos con otros desarrolladores
- Cuando no estás en Python (Node, Go, etc.)
- Debugging de APIs

## 1.6 Comparación de Métodos (Ollama)

### Tabla Comparativa

| Aspecto | LangChain | requests | curl |
|---------|-----------|----------|------|
| **Lenguaje** | Python | Python | Cualquiera (terminal) |
| **Líneas de código** | 3 | 7 | 1 (pero larga) |
| **Facilidad** | ⭐⭐⭐ Muy fácil | ⭐⭐ Medio | ⭐ Requiere conocer terminal |
| **Control** | ⭐ Abstracto | ⭐⭐ Control medio | ⭐⭐⭐ Control total |
| **Para aprender** | ⭐ Uso práctico | ⭐⭐⭐ Entender HTTP | ⭐⭐ Debugging |
| **Para producción** | ⭐⭐⭐ Sí | ⭐⭐ Sí | ❌ No (solo testing) |
| **Debugging** | ⭐ Difícil | ⭐⭐ Medio | ⭐⭐⭐ Fácil |

### Código equivalente lado a lado

**LangChain:**
```python
llm = ChatOllama(model="llama3.2")
response = llm.invoke("pregunta")
print(response.content)
```

**requests:**
```python
response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3.2", "prompt": "pregunta", "stream": False}
)
print(response.json()["response"])
```

**curl:**
```bash
curl -s http://localhost:11434/api/generate \
  -d '{"model":"llama3.2","prompt":"pregunta","stream":false}'
```

### Recomendación por caso de uso

- **Desarrollo de aplicaciones** → LangChain
- **Aprender cómo funcionan las APIs** → requests
- **Testing y debugging rápido** → curl
- **Documentación y ejemplos** → curl (más universal)

## 1.7 Resumen: Patrón Ollama

### ✅ Lo que aprendimos

**Instalación:**
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**Descarga de modelos:**
```bash
ollama pull llama3.2
```

**Servidor:**
- Automático en `localhost:11434`
- En Colab: iniciar con `ollama serve`

**3 métodos de uso:**
- ✅ LangChain: Fácil y rápido
- ✅ requests: Control medio, entender HTTP
- ✅ curl: Testing y debugging

### Cuándo usar Ollama

- ✅ Estás aprendiendo sobre LLMs locales
- ✅ Quieres algo que funcione en 5 minutos
- ✅ No quieres compilar código
- ✅ Necesitas API compatible con OpenAI

### Siguiente paso

**PARTE 2:** Patrón llama-cpp-python (sin servidor HTTP)

---
# PARTE 2: Patrón llama-cpp-python

## 🐍 ¿Qué es llama-cpp-python?

Un **wrapper de Python** para llama.cpp que permite usarlo directamente **sin servidor HTTP**.

### Diferencia clave con Ollama

**Ollama:**
```
Tu script → HTTP → Servidor Ollama (daemon) → llama.cpp → Modelo
```

**llama-cpp-python:**
```
Tu script → llama-cpp-python → Modelo (directo)
```

### Sin servidor = Sin HTTP

| Característica | Ollama | llama-cpp-python |
|----------------|--------|------------------|
| **Servidor HTTP** | ✅ Sí (puerto 11434) | ❌ No |
| **Daemon corriendo** | ✅ Sí (en background) | ❌ No |
| **Uso con curl** | ✅ Sí | ❌ No |
| **Uso con requests** | ✅ Sí | ❌ No |
| **Uso Python directo** | ❌ No (solo HTTP) | ✅ Sí |
| **LangChain** | ✅ Sí | ✅ Sí |

### Cuándo usar llama-cpp-python

- ✅ Scripts que se ejecutan una sola vez
- ✅ No quieres un daemon corriendo en background
- ✅ Solo necesitas Python
- ✅ Quieres control directo del modelo en memoria

## 2.1 Instalación

In [ ]:
# Instalar llama-cpp-python
# Esto puede tardar 2-5 minutos
!pip install -q llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 10.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00


**¿Qué se instaló?**

- Un wrapper de Python que incluye llama.cpp **pre-compilado**
- Binarios optimizados para CPU
- Listo para usar (no requiere compilación)

**Nota sobre GPU:**
- La versión que instalamos funciona en CPU
- Para GPU NVIDIA, necesitas compilar desde fuente
- Para este lab usaremos CPU (funciona en Colab y local)

## 2.2 Descargar Modelo Manualmente

### ¿Por qué manual?

- llama-cpp-python **NO gestiona descargas** como Ollama
- Tú decides qué modelo descargar y dónde guardarlo
- Más control, pero más trabajo

### ¿De dónde descargamos?

- **HuggingFace** → Repositorio de modelos
- **TheBloke** → Usuario especializado en modelos GGUF cuantizados
- Usaremos `wget` para descargar directamente

In [ ]:
%%bash
# Crear directorio para modelos (si no existe)
mkdir -p ./models

# Descargar modelo GGUF desde HuggingFace
# Usamos TinyLlama porque es pequeño (~700MB)
echo "📥 Descargando modelo TinyLlama (esto puede tardar 1-3 minutos)..."
wget -q --show-progress \
  https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf \
  -O ./models/tinyllama.gguf

# Verificar descarga
echo ""
echo "✅ Modelo descargado:"
ls -lh ./models/tinyllama.gguf

📥 Descargando modelo TinyLlama (esto puede tardar 1-3 minutos)...

✅ Modelo descargado:
-rw-r--r-- 1 root root 638M Jan 28 17:04 ./models/tinyllama.gguf



     0K .......... .......... .......... .......... ..........  0% 43.2K 4h12m
    50K .......... .......... .......... .......... ..........  0% 27.8M 2h6m
   100K .......... .......... .......... .......... ..........  0% 66.5M 84m11s
   150K .......... .......... .......... .......... ..........  0% 80.4M 63m10s
   200K .......... .......... .......... .......... ..........  0% 73.5M 50m34s
   250K .......... .......... .......... .......... ..........  0% 88.9M 42m9s
   300K .......... .......... .......... .......... ..........  0% 78.0M 36m9s
   350K .......... .......... .......... .......... ..........  0% 97.7M 31m38s
   400K .......... .......... .......... .......... ..........  0% 88.6M 28m8s
   450K .......... .......... .......... .......... ..........  0%  156K 32m17s
   500K .......... .......... .......... .......... ..........  0% 84.2M 29m21s
   550K .......... .......... .......... .......... ..........  0% 73.9K 39m10s
   600K .......... .......... .......... ....

**Explicación del comando:**

```bash
mkdir -p ./models
```
- `mkdir` → Crear directorio
- `-p` → No dar error si ya existe

```bash
wget -q --show-progress URL -O archivo
```
- `wget` → Descargar archivos de internet
- `-q` → Quiet (menos output)
- `--show-progress` → Mostrar barra de progreso
- `-O` → Output: dónde guardar el archivo

**Sobre el modelo:**
- `TinyLlama-1.1B` → Modelo de 1.1 billones de parámetros
- `.Q4_K_M.gguf` → Cuantizado a 4 bits (25% del tamaño original)
- Tamaño: ~700MB (vs 4GB sin cuantizar)

**¿Por qué TheBloke?**
- Convierte modelos populares a formato GGUF
- Los cuantiza (reduce tamaño)
- Muy confiable en la comunidad

## 2.3 Método 1: LangChain

LangChain tiene soporte para llama-cpp-python con el mismo estilo que otros proveedores.

In [ ]:
# Instalar la integración de LangChain
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_community.llms import LlamaCpp

# 1. Crear cliente (esto carga el modelo en memoria)
print("🔄 Cargando modelo en memoria...")
llm = LlamaCpp(
    model_path="./models/tinyllama.gguf",
    n_ctx=2048,        # Contexto máximo (tokens)
    n_gpu_layers=0,    # 0 = solo CPU
    temperature=0.7,
    verbose=False      # No mostrar logs internos
)
print("✅ Modelo cargado")

# 2. Hacer pregunta (igual que con Ollama)
response = llm.invoke("what is Python?")

# 3. Mostrar
print("\n💬 Respuesta:")
print(response)

🔄 Cargando modelo en memoria...


llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64


✅ Modelo cargado

💬 Respuesta:

Pytheon is a powerful programming language that combines the advantages of other programming languages, such as Python, C++ and Perl. It uses object-oriented programming and functional programming concepts, making it very flexible for solving complex problems. Pytheon can be used to develop various applications from scientific computing to web development.
It has a small learning curve compared with some other languages like Java or C++, and is often considered to be easier to learn than Python. The syntax is simple and intuitive, but it does require some familiarity with programming concepts if you are not already proficient in one or more of them.
Pytheon is an open-source language and has a dedicated community for its development. It is also used as a back-end development tool by the CERN LHC computing group, where it is used to develop various scientific applications such as simulation software for particle physics experiments. Pytheon has been adopt

**Diferencias con ChatOllama:**

| ChatOllama (Ollama) | LlamaCpp (llama-cpp-python) |
|---------------------|-----------------------------|
| `model="llama3.2"` | `model_path="./models/..."` |
| Nombre del modelo | Ruta al archivo |
| Conexión HTTP automática | Carga en memoria |
| Rápido (modelo ya cargado) | Lento primera vez (carga el modelo) |
| No controlas n_ctx | Debes especificar n_ctx |

**Similaridades:**
- `.invoke()` funciona igual
- API de LangChain idéntica
- Puedes usar chains, agents, etc.

**Parámetros importantes:**

```python
model_path="./models/tinyllama.gguf"
```
- Ruta completa al archivo .gguf
- Puede ser absoluta o relativa

```python
n_ctx=2048
```
- Contexto máximo en tokens
- Más = más memoria pero más capacidad

```python
n_gpu_layers=0
```
- Cuántas capas del modelo poner en GPU
- 0 = solo CPU
- 35 = ~todas las capas en GPU (si tienes GPU)

## 2.4 Método 2: Python directo (No hay requests)

### ⚠️ No hay equivalente a requests

**¿Por qué?**
- llama-cpp-python **NO tiene servidor HTTP**
- No hay endpoint para llamar con `requests.post()`
- Solo se usa desde Python directamente

### Alternativa: API nativa de llama-cpp-python

Si no quieres usar LangChain, puedes usar la API nativa:

**Nota importante:** El `pip install llama-cpp-python` que ejecutamos antes ya instaló el módulo `llama_cpp`. La librería se llama `llama-cpp-python` (con guiones) pero el módulo Python se importa como `llama_cpp` (con guión bajo):

```python
# El paquete se instala así:
pip install llama-cpp-python

# Pero se importa así:
from llama_cpp import Llama
```

Esto es una convención común en Python (ej: `scikit-learn` se importa como `sklearn`).

In [ ]:
from llama_cpp import Llama

# 1. Cargar modelo (clase nativa, no LangChain)
print("🔄 Cargando modelo...")
llm_native = Llama(
    model_path="./models/tinyllama.gguf",
    n_ctx=2048,
    verbose=False
)
print("✅ Modelo cargado")

# 2. Generar (API de llama-cpp)
output = llm_native(
    'what is Python?',
    #"¿Qué es Python en una oración?",
    max_tokens=50,
    temperature=0.7
)

# 3. Resultado es un diccionario
print("\n💬 Respuesta:")
print(output["choices"][0]["text"])

🔄 Cargando modelo...
✅ Modelo cargado

💬 Respuesta:
 And what is the purpose of using it?


**Explicación:**

```python
from llama_cpp import Llama
```
- Clase nativa de llama-cpp-python
- No es LangChain

```python
llm_native("texto", max_tokens=50)
```
- Llamada directa al modelo
- Retorna diccionario con formato OpenAI

```python
output["choices"][0]["text"]
```
- Estructura compatible con API de OpenAI
- `choices` → Lista de respuestas
- `[0]` → Primera respuesta
- `text` → El texto generado

**Diferencia con requests:**

| requests (HTTP) | llama-cpp-python |
|-----------------|------------------|
| Llamada por red | Llamada local |
| Servidor separado | Mismo proceso |
| Más lento (red) | Más rápido (memoria) |
| Funciona en cualquier lenguaje | Solo Python |

**Ventaja:**
- Sin overhead de HTTP
- Más rápido que llamar a servidor

**Desventaja:**
- Solo funciona en el mismo script Python
- El modelo se carga cada vez que ejecutas

## 2.5 Método 3: curl (No disponible)

### ❌ No hay curl con llama-cpp-python

**Motivo:**
- No hay servidor HTTP
- curl solo funciona con HTTP
- Es imposible usarlo

**Si quieres usar curl con llama.cpp:**
- Necesitas el patrón **llama.cpp con servidor** (PARTE 3)
- Ese patrón incluye `llama-server` (servidor HTTP)

**Comparación:**

| Componente | Servidor HTTP | Uso con curl |
|------------|---------------|-------------|
| Ollama | ✅ Sí | ✅ Sí |
| llama-cpp-python | ❌ No | ❌ No |
| llama.cpp (compilado) | ✅ Sí (`llama-server`) | ✅ Sí |
| vLLM | ✅ Sí | ✅ Sí |

## 2.6 Resumen: llama-cpp-python

### ✅ Lo que aprendimos

**Instalación:**
```bash
pip install llama-cpp-python
```

**Descarga de modelos:**
```bash
wget URL -O ./models/modelo.gguf
```
- Manual (no automática como Ollama)
- Desde HuggingFace (TheBloke)

**Métodos disponibles:**
- ✅ **LangChain:** Sí (con `LlamaCpp`)
- ❌ **requests:** No (sin servidor)
- ❌ **curl:** No (sin servidor)
- ✅ **Python nativo:** Sí (clase `Llama`)

### Ventajas vs Ollama

✅ Sin daemon en background  
✅ Llamada local (más rápida que HTTP)  
✅ Control directo del modelo  

### Desventajas vs Ollama

❌ Sin servidor HTTP  
❌ Descarga manual de modelos  
❌ Modelo se carga cada ejecución  
❌ Solo Python  

### Cuándo usar llama-cpp-python

- ✅ Scripts que corren una vez
- ✅ No quieres daemon corriendo
- ✅ Solo necesitas Python
- ✅ Quieres máximo control

### Siguiente paso

**PARTE 3:** llama.cpp compilado con servidor HTTP

---
# PARTE 3: Patrón llama.cpp Compilado con Servidor

## 🔧 ¿Qué es llama.cpp compilado?

El **código C++ original de llama.cpp** compilado directamente, que incluye:
- `llama-server` → Servidor HTTP (como Ollama)
- `llama-cli` → Interfaz de línea de comandos
- Máximo control y rendimiento

### Diferencias con llama-cpp-python

**llama-cpp-python:**
```
Tu script Python → Wrapper Python → llama.cpp → Modelo
```

**llama.cpp compilado:**
```
Tu código → HTTP → llama-server (C++ puro) → Modelo
```

### Ventajas del C++ puro

| Aspecto | llama-cpp-python | llama.cpp compilado |
|---------|------------------|---------------------|
| **Lenguaje** | Python wrapper | C++ nativo |
| **Rendimiento** | ⭐⭐ Bueno | ⭐⭐⭐ Máximo |
| **Servidor HTTP** | ❌ No | ✅ Sí (`llama-server`) |
| **Memoria** | Más (Python + C++) | Menos (solo C++) |
| **Compilación** | No necesaria | ✅ Necesaria |
| **Dificultad** | ⭐⭐ Media | ⭐⭐⭐ Alta |

### Cuándo usar llama.cpp compilado

- ✅ Necesitas **máximo rendimiento**
- ✅ Quieres servidor HTTP sin capas extra (sin Ollama)
- ✅ Trabajas en producción con recursos limitados
- ✅ Necesitas control total de compilación (flags, optimizaciones)

## 3.1 Compilación e Instalación

### Versión Simplificada (Solo lo esencial)

Compilaremos llama.cpp con soporte CPU (funciona en cualquier máquina).

In [ ]:
%%bash
# 1. Instalar dependencias para compilación
apt-get update -qq
apt-get install -y -qq build-essential git cmake

# Clean up any previous llama.cpp directory for a fresh build
if [ -d "llama.cpp" ]; then
    echo "🗑️ Limpiando directorio llama.cpp existente..."
    rm -rf llama.cpp
fi

# 2. Clonar repositorio de llama.cpp
echo "📥 Clonando llama.cpp..."
git clone https://github.com/ggerganov/llama.cpp.git -q

# 3. Compilar (solo CPU, con soporte para server) usando CMake
cd llama.cpp
echo "🔨 Compilando llama.cpp con CMake (incluyendo server)...
"
cmake -B build -S . -DLLAMA_BUILD_SERVER=ON
cmake --build build --config Release -j$(nproc)

# 4. Verificar compilación
echo "Current directory after build: $(pwd)"
echo "Contents of build/bin/ directory:"
ls -l build/bin/

if [ -f "build/bin/server" ]; then
    echo "✅ Compilación exitosa: llama-server executable 'server'"
    ls -lh build/bin/server
else
    echo "❌ Error en la compilación: 'server' executable not found in build/bin/"
    exit 1
fi


🗑️ Limpiando directorio llama.cpp existente...
📥 Clonando llama.cpp...
🔨 Compilando llama.cpp con CMake (incluyendo server)...

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
CMAKE_BUILD_TYPE=Release


CalledProcessError: Command 'b'# 1. Instalar dependencias para compilaci\xc3\xb3n\napt-get update -qq\napt-get install -y -qq build-essential git cmake\n\n# Clean up any previous llama.cpp directory for a fresh build\nif [ -d "llama.cpp" ]; then\n    echo "\xf0\x9f\x97\x91\xef\xb8\x8f Limpiando directorio llama.cpp existente..."\n    rm -rf llama.cpp\nfi\n\n# 2. Clonar repositorio de llama.cpp\necho "\xf0\x9f\x93\xa5 Clonando llama.cpp..."\ngit clone https://github.com/ggerganov/llama.cpp.git -q\n\n# 3. Compilar (solo CPU, con soporte para server) usando CMake\ncd llama.cpp\necho "\xf0\x9f\x94\xa8 Compilando llama.cpp con CMake (incluyendo server)...\n"\ncmake -B build -S . -DLLAMA_BUILD_SERVER=ON\ncmake --build build --config Release -j$(nproc)\n\n# 4. Verificar compilaci\xc3\xb3n\necho "Current directory after build: $(pwd)"\necho "Contents of build/bin/ directory:"\nls -l build/bin/\n\nif [ -f "build/bin/server" ]; then\n    echo "\xe2\x9c\x85 Compilaci\xc3\xb3n exitosa: llama-serve

In [ ]:
import subprocess
import time
import requests
import os

# Terminate any existing llama-server process
# This is important to ensure a clean restart
existing_process = None
try:
    # Check if a process is already listening on port 8080
    # This part might need manual intervention or more robust process management in a real scenario
    pass # For simplicity, we assume previous run cleaned up or it will fail and we diagnose
except Exception as e:
    print(f"Warning: Could not check for existing process: {e}")

# Iniciar llama-server en background
print("🚀 Iniciando llama-server en puerto 8080...")

# Verificar que la ruta absoluta del modelo existe
model_path = os.path.abspath("./models/tinyllama.gguf")
print(f"📂 Ruta del modelo: {model_path}")

# Verify server executable path
server_executable_path = "./llama.cpp/build/bin/llama-server" # Corrected executable name
if not os.path.exists(server_executable_path):
    print(f"❌ Error: El ejecutable del servidor no se encontró en {server_executable_path}")
    print("Revisa la celda de compilación (PARTE 3.1) para asegurarte de que se compiló correctamente.")
    # Attempt to list contents of build/bin for more info
    build_bin_dir = os.path.dirname(server_executable_path)
    if os.path.exists(build_bin_dir):
        print(f"Contenido de {build_bin_dir}: {os.listdir(build_bin_dir)}")
    else:
        print(f"El directorio {build_bin_dir} no existe.")
    raise FileNotFoundError(f"No such file or directory: {server_executable_path}")
else:
    print(f"✅ Ejecutable del servidor encontrado en {server_executable_path}")

# Iniciar servidor
server_process = subprocess.Popen(
    [
        server_executable_path,
        "-m", model_path,
        "--port", "8080",
        "--host", "0.0.0.0",
        "-c", "2048",  # Contexto
        "-n", "512"    # Max tokens a generar
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    preexec_fn=os.setsid # Detach from current process group to avoid being killed on notebook restart
)

# Esperar a que inicie
print("⏳ Esperando a que el servidor inicie...")
# Increase sleep time significantly for initial startup and model loading
time.sleep(25) # Increased sleep time for server initialization

# Check for any immediate errors printed by the server
# This will read any output that occurred during the sleep period
stdout_output = server_process.stdout.read().strip()
stderr_output = server_process.stderr.read().strip()

if stdout_output:
    print("--- llama-server stdout ---")
    print(stdout_output)
    print("---------------------------")
if stderr_output:
    print("--- llama-server stderr ---")
    print(stderr_output)
    print("---------------------------")

# Verificar que está corriendo con el endpoint compatible con OpenAI
try:
    # Try the /health endpoint first for a basic check
    health_response = requests.get("http://localhost:8080/health", timeout=10)
    if health_response.status_code == 200:
        print("✅ Servidor llama-server está vivo en localhost:8080/health")
        # Then try the /v1/models endpoint
        response = requests.get("http://localhost:8080/v1/models", timeout=10)
        if response.status_code == 200 and response.json() and len(response.json().get('data', [])) > 0:
            print("✅ Servidor llama-server está corriendo en localhost:8080")
            print(f"📦 Modelo cargado: {response.json()['data'][0]['id']}")
        else:
            print(f"⚠️ Servidor responde pero con código: {response.status_code} en /v1/models o no tiene modelos cargados.")
            print("Intenta ejecutar esta celda nuevamente si el servidor no está listo.")
    else:
        print(f"⚠️ Servidor no responde a /health con código 200. Código: {health_response.status_code}")
        print("Intenta ejecutar esta celda nuevamente")
except requests.exceptions.ConnectionError:
    print(f"❌ Error al conectar al servidor en localhost:8080. Es posible que el servidor no haya iniciado correctamente o haya fallado.")
    print("Revisa los logs del servidor arriba para más detalles.")
    print("Intenta ejecutar esta celda nuevamente")
except Exception as e:
    print(f"❌ Error al verificar servidor: {e}")
    print("Intenta ejecutar esta celda nuevamente")

**Explicación del proceso:**

```bash
apt-get install build-essential git cmake
```
- `build-essential` → Compilador C++ (g++, make)
- `git` → Para clonar el repositorio
- `cmake` → Sistema de construcción (aunque usamos make directamente)

```bash
git clone https://github.com/ggerganov/llama.cpp.git
```
- Descarga el código fuente oficial
- Crea directorio `llama.cpp/`

```bash
make llama-server -j$(nproc)
```
- `make` → Compila el código C++
- `llama-server` → Target específico (solo compilamos el servidor)
- `-j$(nproc)` → Usa todos los cores de CPU (paraleliza compilación)

**¿Qué se compiló?**
- Ejecutable `llama-server` (servidor HTTP en C++ puro)
- Tamaño: ~7-10 MB (muy ligero)

## 3.2 Descargar Modelo

Usaremos el mismo modelo que descargamos para llama-cpp-python (si no lo tienes, descárgalo nuevamente).

In [ ]:
%%bash
# Verificar si el modelo ya existe
if [ -f "./models/tinyllama.gguf" ]; then
    echo "✅ Modelo ya descargado:"
    ls -lh ./models/tinyllama.gguf
else
    echo "📥 Descargando modelo TinyLlama..."
    mkdir -p ./models
    wget -q --show-progress \
      https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf \
      -O ./models/tinyllama.gguf
    echo "✅ Modelo descargado"
fi

📥 Descargando modelo TinyLlama...
✅ Modelo descargado



     0K .......... .......... .......... .......... ..........  0% 47.9K 3h47m
    50K .......... .......... .......... .......... ..........  0% 5.31M 1h54m
   100K .......... .......... .......... .......... ..........  0% 3.73M 77m22s
   150K .......... .......... .......... .......... ..........  0% 1.47M 59m50s
   200K .......... .......... .......... .......... ..........  0% 4.83M 48m18s
   250K .......... .......... .......... .......... ..........  0% 4.04M 40m41s
   300K .......... .......... .......... .......... ..........  0% 2.19M 35m34s
   350K .......... .......... .......... .......... ..........  0% 5.74M 31m21s
   400K .......... .......... .......... .......... ..........  0% 5.28M 28m5s
   450K .......... .......... .......... .......... ..........  0% 5.63M 25m28s
   500K .......... .......... .......... .......... ..........  0% 6.71M 23m17s
   550K .......... .......... .......... .......... ..........  0% 8.51M 21m27s
   600K .......... .......... .......... .

## 3.3 Iniciar Servidor llama-server

El servidor HTTP de llama.cpp se llama `llama-server` y es compatible con la API de OpenAI.

In [ ]:
import subprocess
import time
import requests
import os

# Terminate any existing llama-server process
# This is important to ensure a clean restart
existing_process = None
try:
    # Check if a process is already listening on port 8080
    # This part might need manual intervention or more robust process management in a real scenario
    pass # For simplicity, we assume previous run cleaned up or it will fail and we diagnose
except Exception as e:
    print(f"Warning: Could not check for existing process: {e}")

# Iniciar llama-server en background
print("🚀 Iniciando llama-server en puerto 8080...")

# Verificar que la ruta absoluta del modelo existe
model_path = os.path.abspath("./models/tinyllama.gguf")
print(f"📂 Ruta del modelo: {model_path}")

# Iniciar servidor
server_process = subprocess.Popen(
    [
        "./llama.cpp/build/bin/server", # Corrected path from llama-server to server
        "-m", model_path,
        "--port", "8080",
        "--host", "0.0.0.0",
        "-c", "2048",  # Contexto
        "-n", "512"    # Max tokens a generar
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    preexec_fn=os.setsid # Detach from current process group to avoid being killed on notebook restart
)

# Esperar a que inicie
print("⏳ Esperando a que el servidor inicie...")
# Increase sleep time significantly for initial startup and model loading
time.sleep(25) # Increased sleep time for server initialization

# Check for any immediate errors printed by the server
# This will read any output that occurred during the sleep period
stdout_output = server_process.stdout.read().strip()
stderr_output = server_process.stderr.read().strip()

if stdout_output:
    print("--- llama-server stdout ---")
    print(stdout_output)
    print("---------------------------")
if stderr_output:
    print("--- llama-server stderr ---")
    print(stderr_output)
    print("---------------------------")

# Verificar que está corriendo con el endpoint compatible con OpenAI
try:
    # Try the /health endpoint first for a basic check
    health_response = requests.get("http://localhost:8080/health", timeout=10)
    if health_response.status_code == 200:
        print("✅ Servidor llama-server está vivo en localhost:8080/health")
        # Then try the /v1/models endpoint
        response = requests.get("http://localhost:8080/v1/models", timeout=10)
        if response.status_code == 200 and response.json() and len(response.json().get('data', [])) > 0:
            print("✅ Servidor llama-server está corriendo en localhost:8080")
            print(f"📦 Modelo cargado: {response.json()['data'][0]['id']}")
        else:
            print(f"⚠️ Servidor responde pero con código: {response.status_code} en /v1/models o no tiene modelos cargados.")
            print("Intenta ejecutar esta celda nuevamente si el servidor no está listo.")
    else:
        print(f"⚠️ Servidor no responde a /health con código 200. Código: {health_response.status_code}")
        print("Intenta ejecutar esta celda nuevamente")
except requests.exceptions.ConnectionError:
    print(f"❌ Error al conectar al servidor en localhost:8080. Es posible que el servidor no haya iniciado correctamente o haya fallado.")
    print("Revisa los logs del servidor arriba para más detalles.")
    print("Intenta ejecutar esta celda nuevamente")
except Exception as e:
    print(f"❌ Error al verificar servidor: {e}")
    print("Intenta ejecutar esta celda nuevamente")


**Parámetros del servidor:**

```bash
llama-server -m modelo.gguf --port 8080
```
- `-m` → Ruta al modelo .gguf
- `--port 8080` → Puerto del servidor (Ollama usa 11434)
- `--host 0.0.0.0` → Escuchar en todas las interfaces
- `-c 2048` → Contexto máximo
- `-n 512` → Máximo de tokens a generar por respuesta

**Endpoints disponibles:**
- `GET /health` → Verificar estado del servidor
- `POST /completion` → Generar texto (API de llama.cpp)
- `POST /v1/chat/completions` → API compatible con OpenAI

**Diferencias con Ollama:**

| Ollama | llama-server |
|--------|--------------|
| Puerto 11434 | Puerto 8080 (configurable) |
| API propia + OpenAI | API OpenAI + llama.cpp |
| Gestión automática | Manual (tú inicias/detienes) |
| Múltiples modelos | Un modelo a la vez |

## 3.4 Método 1: LangChain con OpenAI SDK

Como llama-server es compatible con la API de OpenAI, usamos `langchain-openai`.

In [ ]:
# Instalar LangChain con soporte OpenAI
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.7 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI

# Conectar a llama-server usando API de OpenAI
llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",  # llama-server endpoint
    api_key="dummy",  # llama-server no requiere key, pero LangChain lo pide
    model="local-model",  # Nombre ficticio (llama-server solo sirve un modelo)
    temperature=0.7
)

# Hacer pregunta
response = llm.invoke("¿Qué es Python en una oración?")

print("💬 Respuesta:")
print(response.content)

**Explicación:**

```python
from langchain_openai import ChatOpenAI
```
- Usamos `ChatOpenAI` porque llama-server es compatible con API de OpenAI
- NO usamos `ChatOllama` ni `LlamaCpp`

```python
base_url="http://localhost:8080/v1"
```
- URL del servidor llama-server
- `/v1` → Prefijo de la API de OpenAI
- Diferente al puerto de Ollama (11434)

```python
api_key="dummy"
```
- llama-server NO requiere autenticación
- Pero LangChain exige el parámetro, así que ponemos cualquier valor

```python
model="local-model"
```
- Nombre ficticio (puede ser cualquier string)
- llama-server solo sirve el modelo que iniciaste con `-m`

**Ventaja de usar API de OpenAI:**
- Código compatible con producción (solo cambias `base_url`)
- Mismo código funciona para OpenAI, llama-server, vLLM, etc.

## 3.5 Método 2: requests

Usamos el endpoint `/v1/chat/completions` compatible con OpenAI.

In [ ]:
import requests

# Endpoint compatible con OpenAI
url = "http://localhost:8080/v1/chat/completions"

# Payload en formato OpenAI Chat
payload = {
    "model": "local-model",
    "messages": [
        {"role": "user", "content": "¿Qué es Python en una oración?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
}

# Hacer petición
response = requests.post(url, json=payload)
result = response.json()

# Extraer respuesta
print("💬 Respuesta:")
print(result["choices"][0]["message"]["content"])

**Diferencias con Ollama:**

**Ollama:**
```python
payload = {
    "model": "llama3.2",
    "prompt": "pregunta",
    "stream": False
}
```

**llama-server (OpenAI format):**
```python
payload = {
    "model": "local-model",
    "messages": [{"role": "user", "content": "pregunta"}],
    "temperature": 0.7
}
```

| Aspecto | Ollama | llama-server |
|---------|--------|--------------|
| **Formato** | Propio de Ollama | OpenAI Chat API |
| **Campo de entrada** | `prompt` (string) | `messages` (lista) |
| **Endpoint** | `/api/generate` | `/v1/chat/completions` |
| **Respuesta** | `result["response"]` | `result["choices"][0]["message"]["content"]` |

**Ventaja del formato OpenAI:**
- Estándar de la industria
- Compatible con múltiples proveedores (OpenAI, Azure, vLLM, etc.)
- Soporte para conversaciones multi-turn (system, user, assistant)

## 3.6 Método 3: curl

In [ ]:
%%bash
curl -s http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "local-model",
    "messages": [
      {"role": "user", "content": "¿Qué es Python en una oración?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
  }' \
  | python3 -m json.tool

**Diferencia con Ollama curl:**

**Ollama:**
```bash
curl http://localhost:11434/api/generate \
  -d '{"model":"llama3.2","prompt":"pregunta","stream":false}'
```

**llama-server:**
```bash
curl http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "local-model",
    "messages": [{"role": "user", "content": "pregunta"}]
  }'
```

**Nota sobre `-H "Content-Type: application/json"`:**
- En Ollama no es necesario (lo asume por defecto)
- En llama-server (API OpenAI) es recomendable especificarlo
- Indica que el cuerpo es JSON

## 3.7 Resumen: llama.cpp Compilado

### ✅ Lo que aprendimos

**Instalación:**
```bash
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp
make llama-server
```

**Iniciar servidor:**
```bash
./llama-server -m modelo.gguf --port 8080
```

**3 métodos de uso:**
- ✅ **LangChain:** `ChatOpenAI` con `base_url` personalizada
- ✅ **requests:** Endpoint `/v1/chat/completions` (OpenAI format)
- ✅ **curl:** Compatible con estándar OpenAI

### Ventajas vs Ollama

✅ **Máximo rendimiento** (C++ puro, sin capas extra)  
✅ **API estándar OpenAI** (más portable)  
✅ **Control total** de compilación y flags  
✅ **Menor consumo de memoria**  

### Desventajas vs Ollama

❌ Requiere compilación  
❌ Gestión manual del servidor  
❌ Solo un modelo a la vez  
❌ Sin gestión automática de modelos  

### Ventajas vs llama-cpp-python

✅ Servidor HTTP (accesible desde cualquier lenguaje)  
✅ Más eficiente (sin wrapper Python)  
✅ API estándar (OpenAI)  

### Cuándo usar llama.cpp compilado

- ✅ Necesitas **máximo rendimiento**
- ✅ Quieres servidor HTTP sin overhead de Ollama
- ✅ Trabajas en producción con recursos limitados
- ✅ Necesitas API compatible con OpenAI para portabilidad

### Siguiente paso

**PARTE 4:** vLLM (Producción con GPU NVIDIA)

---
# PARTE 4: Patrón vLLM (Producción con GPU)

## 🚀 ¿Qué es vLLM?

Un **motor de inferencia de alto rendimiento** desarrollado por UC Berkeley para **producción**.

### Características principales

- 🔥 **Optimizado para GPU NVIDIA** (pero funciona en CPU)
- ⚡ **PagedAttention**: Gestión eficiente de memoria
- 🔄 **Continuous batching**: Procesa múltiples requests simultáneamente
- 📊 **Hasta 24x más rápido** que PyTorch vanilla
- 🌐 **API compatible con OpenAI**

### Comparación con otros patrones

| Aspecto | Ollama | llama.cpp | vLLM |
|---------|--------|-----------|------|
| **Optimizado para** | CPU + GPU | CPU (principalmente) | GPU (NVIDIA) |
| **Rendimiento GPU** | ⭐⭐ Bueno | ⭐ Básico | ⭐⭐⭐ Máximo |
| **Batching** | ❌ No | ❌ No | ✅ Sí (continuous) |
| **Memoria GPU** | ⭐⭐ Estándar | ⭐⭐ Estándar | ⭐⭐⭐ PagedAttention |
| **Producción** | ⭐⭐ Desarrollo | ⭐⭐ Medio | ⭐⭐⭐ Producción |
| **Facilidad** | ⭐⭐⭐ Muy fácil | ⭐⭐ Medio | ⭐⭐ Medio |

### Cuándo usar vLLM

- ✅ Tienes **GPU NVIDIA** (T4, A10, A100, etc.)
- ✅ Vas a **producción** con alto tráfico
- ✅ Necesitas **máximo throughput**
- ✅ Quieres **servir múltiples requests** simultáneamente
- ✅ Necesitas **API compatible con OpenAI**

### Cuándo NO usar vLLM

- ❌ Solo tienes CPU (usa Ollama o llama.cpp)
- ❌ Estás aprendiendo (Ollama es más simple)
- ❌ Un solo usuario (overhead innecesario)

### Nota sobre Google Colab y GPU

**Colab gratuito SÍ tiene GPU NVIDIA** (generalmente T4 con 15GB VRAM):
- Para activar GPU: `Runtime > Change runtime type > GPU`
- La GPU no está garantizada (depende de disponibilidad)
- Colab Pro: GPU garantizada y con más tiempo de sesión

**Cómo verificar si tienes GPU en Colab:**
```python
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nombre: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
```

vLLM detecta automáticamente si tienes GPU y la usa.

## 4.1 Instalación de vLLM

In [ ]:
# Instalar vLLM
# Versión CPU (para Colab gratuito y entornos sin GPU NVIDIA)
# Para GPU, simplemente: pip install vllm
!pip install -q vllm

print("✅ vLLM instalado")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 827.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 

**¿Qué se instaló?**

- Motor vLLM con todas las optimizaciones
- Compatible con modelos de HuggingFace
- Servidor HTTP incluido (`vllm serve`)

**Versiones:**
- **CPU**: `pip install vllm` (funciona en cualquier máquina)
- **GPU NVIDIA**: Automáticamente usa CUDA si está disponible

## 4.2 Descargar Modelo (Automático)

A diferencia de llama.cpp, vLLM **descarga modelos automáticamente** desde HuggingFace.

### Modelos compatibles

vLLM funciona con modelos en formato **HuggingFace** (no GGUF):
- Llama, Llama 2, Llama 3
- Mistral, Mixtral
- Qwen, Phi
- Muchos más...

Usaremos `TinyLlama` porque es pequeño (~2.2GB).

## 4.3 Iniciar Servidor vLLM

vLLM incluye `vllm serve` que inicia un servidor HTTP compatible con OpenAI.

In [ ]:
vllm_process = subprocess.Popen(
    [
        "vllm", "serve",
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # Modelo de HuggingFace
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "float16"  # Precisión (float16 para ahorrar memoria)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

response = requests.get("http://localhost:8000/v1/models", timeout=10)
response.status_code


200

In [ ]:
import subprocess
import time
import requests

# Iniciar servidor vLLM en background
print("🚀 Iniciando servidor vLLM (esto descargará el modelo si no existe)...")
print("⏳ Primera vez puede tardar 3-5 minutos (descarga + inicialización)")

# Iniciar servidor
vllm_process = subprocess.Popen(
    [
        "vllm", "serve",
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # Modelo de HuggingFace
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "float16"  # Precisión (float16 para ahorrar memoria)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Esperar a que inicie (más tiempo que otros porque descarga el modelo)
print("⏳ Esperando a que el servidor inicie...")
time.sleep(30)  # vLLM tarda más en iniciar

# Verificar que está corriendo
try:
    response = requests.get("http://localhost:8000/v1/models", timeout=10)
    if response.status_code == 200:
        print("✅ Servidor vLLM está corriendo en localhost:8000")
        print(f"📦 Modelo cargado: {response.json()['data'][0]['id']}")
    else:
        print(f"⚠️ Servidor responde pero con código: {response.status_code}")
except Exception as e:
    print(f"❌ Error al conectar: {e}")
    print("El servidor puede estar aún iniciando. Espera 1-2 minutos más.")

🚀 Iniciando servidor vLLM (esto descargará el modelo si no existe)...
⏳ Primera vez puede tardar 3-5 minutos (descarga + inicialización)
⏳ Esperando a que el servidor inicie...
❌ Error al conectar: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/models (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ff5a8959df0>: Failed to establish a new connection: [Errno 111] Connection refused'))
El servidor puede estar aún iniciando. Espera 1-2 minutos más.


**Parámetros del servidor:**

```bash
vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0
```
- `vllm serve` → Comando para iniciar servidor HTTP
- `TinyLlama/...` → Modelo de HuggingFace (formato `usuario/modelo`)
- Se descarga automáticamente si no existe

```bash
--port 8000
```
- Puerto del servidor (por defecto 8000)
- Diferente de Ollama (11434) y llama-server (8080)

```bash
--dtype float16
```
- Precisión de punto flotante
- `float16` → Ahorra 50% de memoria vs `float32`
- En GPU: mejor rendimiento con `float16`

**¿Dónde se descarga el modelo?**
- Linux: `~/.cache/huggingface/hub/`
- macOS: `~/.cache/huggingface/hub/`
- Windows: `C:\Users\<usuario>\.cache\huggingface\hub\`

**Formato del modelo:**
- vLLM usa **formato HuggingFace** (safetensors, pytorch)
- NO usa GGUF (como llama.cpp y Ollama)

## 4.4 Método 1: LangChain con OpenAI SDK

vLLM es 100% compatible con la API de OpenAI, igual que llama-server.

In [ ]:
from langchain_openai import ChatOpenAI

# Conectar a vLLM (mismo código que llama-server, solo cambia el puerto)
llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # Nombre completo del modelo
    temperature=0.7
)

# Hacer pregunta
response = llm.invoke("¿Qué es Python en una oración?")

print("💬 Respuesta:")
print(response.content)

💬 Respuesta:
Python en una oración significa que Python se encargará de la ejecución de la instrucción, no se necesita declarar una función o método en la instrucción. En Python, las instrucciones se escriben en una expresión y la ejecución se realiza en la función de la instrucción.

En muchos casos, el uso de Python en una oración está ligado al patrón de programación de "Declarando la función y llamándola". Requiere que la función sea declarada y luego se llame en una instrucción (instrucción anónima) o se cree un objeto y luego se llama en una instrucción (instrucción anónima).

Para utilizar Python en una oración, debes declarar una función y llamarla en la instrucción anónima. Cada función Python es una instancia de una clase y puede tener varias funciones.

### Exercise: Writing a Python program

1. Create a Python program that calculates the sum of all elements in a given list.

2. Use a function to calculate the sum.

3. Include comments in your code explaining what the functi

**Portabilidad del código:**

Este mismo código funciona con:

| Proveedor | base_url | model | api_key |
|-----------|----------|-------|---------|
| **OpenAI** | (default) | `gpt-4` | Tu API key real |
| **llama-server** | `http://localhost:8080/v1` | Cualquiera | `dummy` |
| **vLLM** | `http://localhost:8000/v1` | Nombre HF | `dummy` |
| **Azure OpenAI** | Tu endpoint | Deployment name | Tu key |

**Ventaja:**
- Desarrollas localmente con vLLM
- Despliegas a OpenAI cambiando solo 2 líneas

## 4.5 Método 2: requests

In [ ]:
import requests

# Endpoint (idéntico a llama-server, solo cambia el puerto)
url = "http://localhost:8000/v1/chat/completions"

# Payload en formato OpenAI
payload = {
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [
        {"role": "user", "content": "¿Qué es Python en una oración?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
}

# Hacer petición
response = requests.post(url, json=payload)
result = response.json()

# Extraer respuesta
print("💬 Respuesta:")
print(result["choices"][0]["message"]["content"])

💬 Respuesta:
Python en una oración significa "programación en Python". "Python", en sí, es una lenguaje de programación interpretado y es el lenguaje de programación más utilizado para la creación de aplicaciones web. A diferencia de otros lenguajes de programación que utilizan un motor de interprete, Python necesita un entorno de ejecución para ejecutar las instrucciones de código. Los entornos de ejecución para Python incluyen el


**Nota:**

El código es **idéntico** a llama-server, solo cambia:
- Puerto: `8000` (vLLM) vs `8080` (llama-server)
- Nombre del modelo: debe coincidir con el que iniciaste en el servidor

## 4.6 Método 3: curl

In [ ]:
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [
      {"role": "user", "content": "¿Qué es Python en una oración?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
  }' \
  | python3 -m json.tool

{
    "id": "chatcmpl-b83cae5d6cd90e18",
    "object": "chat.completion",
    "created": 1769660026,
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "Python en una oraci\u00f3n es una palabra reservada que se utiliza para indicar que una funci\u00f3n, variable o operaci\u00f3n posee el mismo nombre en Python como en cualquier otro lenguaje de programaci\u00f3n con la misma sintaxis. Algunos ejemplos son:\n\n- Funci\u00f3n `my_function`\n- Variable `my_variable`\n- Operaci\u00f3n `my_operation`\n\nLa palabra reservada `python` se utiliza en",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [],
                "reasoning": null,
                "reasoning_content": null
            },
            "logprobs": null,
            "

## 4.7 Resumen: vLLM

### ✅ Lo que aprendimos

**Instalación:**
```bash
pip install vllm
```

**Iniciar servidor:**
```bash
vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0 --port 8000
```

**Descarga de modelos:**
- Automática desde HuggingFace
- Formato: SafeTensors / PyTorch (NO GGUF)

**3 métodos de uso:**
- ✅ **LangChain:** `ChatOpenAI` con `base_url="http://localhost:8000/v1"`
- ✅ **requests:** Endpoint `/v1/chat/completions` (OpenAI format)
- ✅ **curl:** Idéntico a llama-server, solo cambia puerto

### Ventajas únicas de vLLM

✅ **PagedAttention** → Gestión eficiente de memoria GPU  
✅ **Continuous batching** → Procesa múltiples requests simultáneamente  
✅ **Hasta 24x más rápido** que PyTorch vanilla  
✅ **Optimizado para producción** con alto tráfico  
✅ **API estándar OpenAI** (máxima portabilidad)  

### Ventajas vs Ollama

✅ Mejor rendimiento en GPU NVIDIA  
✅ Continuous batching (más throughput)  
✅ Optimizado para producción  

### Ventajas vs llama.cpp

✅ Mejor rendimiento en GPU (optimizaciones CUDA)  
✅ Continuous batching  
✅ Gestión automática de modelos HuggingFace  

### Desventajas vs Ollama/llama.cpp

❌ Más consumo de recursos (overhead de optimizaciones)  
❌ Enfocado en GPU (CPU funciona pero no es óptimo)  
❌ No usa GGUF (solo HuggingFace format)  

### Cuándo usar vLLM

- ✅ Tienes **GPU NVIDIA**
- ✅ Vas a **producción** con múltiples usuarios
- ✅ Necesitas **máximo throughput**
- ✅ Quieres **API compatible con OpenAI**
- ✅ Necesitas **procesar múltiples requests** simultáneamente

### Siguiente paso

**PARTE 5:** Comparación final de todos los patrones

---
# PARTE 5: Patrón TGI (Alternativa de Producción)

## 🤗 ¿Qué es TGI (Text Generation Inference)?

Un **servidor de inferencia de alto rendimiento** desarrollado por **HuggingFace** para producción.

### TGI vs vLLM: Los dos gigantes

Ambos resuelven el **mismo problema** (servir LLMs en producción), pero con enfoques diferentes:

| Aspecto | vLLM | TGI |
|---------|------|-----|
| **Creador** | UC Berkeley | HuggingFace |
| **Lenguaje** | Python + CUDA | Rust + Python |
| **Optimización estrella** | PagedAttention | Flash Attention |
| **Docker oficial** | ⭐⭐ Bueno | ⭐⭐⭐ Excelente |
| **Integración HuggingFace** | Buena | Nativa |
| **API** | OpenAI compatible | OpenAI compatible |

### ¿Qué es PagedAttention vs Flash Attention?

**PagedAttention (vLLM):**
- Optimiza el **uso de memoria GPU**
- Asigna memoria solo cuando se necesita (como un valet parking inteligente)
- Resultado: **más requests simultáneos** en la misma GPU

```
SIN PagedAttention:              CON PagedAttention:
┌─────────────────────┐          ┌─────────────────────┐
│ Req1: [████░░░░░░░] │          │ Req1: [████]        │
│ Req2: [███░░░░░░░░] │          │ Req2: [███]         │
│ Req3: [NO CABE]     │          │ Req3: [█████]       │
│ (memoria reservada  │          │ Req4: [██]          │
│  pero no usada)     │          │ (memoria compartida)│
└─────────────────────┘          └─────────────────────┘
```

**Flash Attention (TGI):**
- Optimiza la **velocidad de cálculo**
- Minimiza movimiento de datos entre GPU memoria ↔ GPU cómputo
- Resultado: **inferencia más rápida**, menos memoria

```
SIN Flash Attention:
GPU Memoria ←─────────────────→ GPU Cómputo
            (muchos viajes lentos)

CON Flash Attention:
GPU Memoria ←→ [Caché local] ←→ GPU Cómputo
               (todo en bloques rápidos)
```

### ¿Por qué Rust?

TGI usa **Rust** para las partes críticas de rendimiento:

| Lenguaje | Velocidad | Seguridad | Usado por |
|----------|-----------|-----------|-----------|
| Python | ⭐ | ⭐ | Prototipado |
| C++ | ⭐⭐⭐ | ⭐⭐ | llama.cpp |
| **Rust** | ⭐⭐⭐ | ⭐⭐⭐ | TGI |

Rust = velocidad de C++ + seguridad de memoria garantizada.

### Cuándo usar TGI

- ✅ Ya usas mucho **HuggingFace** (integración nativa)
- ✅ Prefieres **Docker plug-and-play**
- ✅ Necesitas **soporte empresarial** (HuggingFace)
- ✅ Quieres **Flash Attention** para máxima velocidad

### Cuándo usar vLLM en lugar de TGI

- ✅ Necesitas **PagedAttention** específicamente
- ✅ Tu equipo ya conoce vLLM
- ✅ Prefieres instalación sin Docker

## 5.1 Instalación de TGI

### Opción A: Docker (Recomendado)

Docker es la forma más fácil de usar TGI. Todo viene preconfigurado.

In [ ]:
%%bash
# Verificar si Docker está instalado
if command -v docker &> /dev/null; then
    echo "✅ Docker está instalado:"
    docker --version
else
    echo "❌ Docker no está instalado"
    echo "Instalar Docker: https://docs.docker.com/get-docker/"
    echo ""
    echo "En Colab/Linux puedes instalar Docker así:"
    echo "curl -fsSL https://get.docker.com | sh"
fi

❌ Docker no está instalado
Instalar Docker: https://docs.docker.com/get-docker/

En Colab/Linux puedes instalar Docker así:
curl -fsSL https://get.docker.com | sh


In [ ]:
!curl -fsSL https://get.docker.com | sh

# Executing docker install script, commit: f381ee68b32e515bb4dc034b339266aff1fbc460
+ sh -c apt-get -qq update >/dev/null
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu jammy stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
+ sh -c DEBIAN_

### Iniciar TGI con Docker (GPU)

```bash
# Con GPU NVIDIA (producción)
docker run --gpus all -p 8080:80 \
  ghcr.io/huggingface/text-generation-inference:latest \
  --model-id TinyLlama/TinyLlama-1.1B-Chat-v1.0

# Sin GPU (solo CPU - más lento)
docker run -p 8080:80 \
  ghcr.io/huggingface/text-generation-inference:latest \
  --model-id TinyLlama/TinyLlama-1.1B-Chat-v1.0
```

**Explicación:**
- `--gpus all` → Usa todas las GPUs disponibles
- `-p 8080:80` → Puerto 8080 en tu máquina → puerto 80 del container
- `--model-id` → Modelo de HuggingFace (se descarga automáticamente)

### Opción B: Sin Docker (más complejo)

Si no quieres usar Docker, puedes instalar TGI directamente:

In [ ]:
# Opción B: Instalación sin Docker (requiere Rust)
# Esto puede tardar varios minutos

# 1. Instalar Rust (si no lo tienes)
# !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# 2. Instalar TGI desde pip
!pip install -q text-generation

print("✅ Cliente TGI instalado")
print("Nota: Para el servidor TGI sin Docker, necesitas compilar desde source")
print("Ver: https://github.com/huggingface/text-generation-inference")

## 5.2 Iniciar Servidor TGI

**Nota importante:** Para este notebook, asumimos que TGI está corriendo en Docker.

Si tienes Docker y GPU, ejecuta en una terminal:
```bash
docker run --gpus all -p 8080:80 \
  ghcr.io/huggingface/text-generation-inference:latest \
  --model-id TinyLlama/TinyLlama-1.1B-Chat-v1.0
```

Si no tienes Docker o GPU, puedes usar la **API de HuggingFace Inference** (cloud) como alternativa:

In [ ]:
# Alternativa: Usar HuggingFace Inference API (cloud, sin instalar nada)
# Requiere cuenta gratuita en huggingface.co

from huggingface_hub import InferenceClient

# Cliente para API de HuggingFace (gratuito con límites)
client = InferenceClient("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Generar texto
response = client.text_generation(
    "¿Qué es Python en una oración?",
    max_new_tokens=50
)

print("💬 Respuesta (HuggingFace Inference API):")
print(response)

## 5.3 Método 1: LangChain con OpenAI SDK

TGI es 100% compatible con la API de OpenAI (igual que vLLM).

**Nota:** Este código requiere que TGI esté corriendo en `localhost:8080`.

In [ ]:
from langchain_openai import ChatOpenAI

# Conectar a TGI (mismo código que vLLM, solo cambia el puerto)
llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",  # Puerto de TGI
    api_key="dummy",  # TGI no requiere API key
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    temperature=0.7
)

# Hacer pregunta
response = llm.invoke("¿Qué es Python en una oración?")

print("💬 Respuesta:")
print(response.content)

## 5.4 Método 2: requests

In [ ]:
import requests

# Endpoint TGI (compatible con OpenAI)
url = "http://localhost:8080/v1/chat/completions"

# Payload en formato OpenAI
payload = {
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [
        {"role": "user", "content": "¿Qué es Python en una oración?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
}

# Hacer petición
response = requests.post(url, json=payload)
result = response.json()

# Extraer respuesta
print("💬 Respuesta:")
print(result["choices"][0]["message"]["content"])

## 5.5 Método 3: curl

In [ ]:
%%bash
curl -s http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [
      {"role": "user", "content": "¿Qué es Python en una oración?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
  }' \
  | python3 -m json.tool

## 5.6 Resumen: TGI

### ✅ Lo que aprendimos

**Instalación (Docker - recomendado):**
```bash
docker run --gpus all -p 8080:80 \
  ghcr.io/huggingface/text-generation-inference:latest \
  --model-id TinyLlama/TinyLlama-1.1B-Chat-v1.0
```

**Descarga de modelos:**
- Automática desde HuggingFace
- Formato: SafeTensors / PyTorch (igual que vLLM)

**3 métodos de uso:**
- ✅ **LangChain:** `ChatOpenAI` con `base_url="http://localhost:8080/v1"`
- ✅ **requests:** Endpoint `/v1/chat/completions` (OpenAI format)
- ✅ **curl:** Idéntico a vLLM, solo cambia puerto

### TGI vs vLLM: Resumen final

| Aspecto | TGI | vLLM |
|---------|-----|------|
| **Optimización** | Flash Attention | PagedAttention |
| **Docker** | ⭐⭐⭐ Excelente | ⭐⭐ Bueno |
| **Sin Docker** | Difícil (Rust) | Fácil (pip) |
| **Integración HF** | Nativa | Buena |
| **API** | OpenAI compat | OpenAI compat |
| **Rendimiento** | ⭐⭐⭐ | ⭐⭐⭐ |

### Cuándo usar TGI

- ✅ Ya usas Docker en tu stack
- ✅ Usas mucho HuggingFace
- ✅ Quieres Flash Attention
- ✅ Necesitas soporte empresarial (HuggingFace)

### Cuándo usar vLLM

- ✅ Prefieres instalación sin Docker
- ✅ Necesitas PagedAttention específicamente
- ✅ Tu equipo ya lo conoce

### Conclusión

**TGI y vLLM son intercambiables** para la mayoría de casos. La diferencia de rendimiento es mínima en la práctica. Elige según:
- **Docker fácil** → TGI
- **pip install** → vLLM
- **Ya usas HuggingFace** → TGI
- **Ya conoces vLLM** → vLLM

### Siguiente paso

**PARTE 6:** Comparación final de todos los patrones (ahora con 5 patrones)

---
# PARTE 6: Comparación Final y Guía de Decisión

## 📊 Tabla Comparativa Completa (5 Patrones)

### 6.1 Comparación por Características

| Característica | Ollama | llama-cpp-python | llama.cpp | vLLM | TGI |
|----------------|--------|------------------|-----------|------|-----|
| **Servidor HTTP** | ✅ Auto | ❌ No | ✅ Manual | ✅ Manual | ✅ Docker |
| **Puerto** | 11434 | N/A | 8080 | 8000 | 8080 |
| **Instalación** | 1 comando | pip | Compilar | pip | Docker |
| **Descarga modelos** | Auto | Manual | Manual | Auto | Auto |
| **Formato modelos** | GGUF | GGUF | GGUF | HuggingFace | HuggingFace |
| **API compatible** | Ollama+OpenAI | N/A | OpenAI | OpenAI | OpenAI |
| **Uso con curl** | ✅ | ❌ | ✅ | ✅ | ✅ |
| **LangChain** | `ChatOllama` | `LlamaCpp` | `ChatOpenAI` | `ChatOpenAI` | `ChatOpenAI` |
| **Optimización** | General | CPU | CPU | PagedAttention | FlashAttention |
| **Mejor para** | Aprender | Scripts | Control | Producción | Producción |
| **Dificultad** | ⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ |

### 6.2 Comparación de Rendimiento

| Aspecto | Ollama | llama-cpp-python | llama.cpp | vLLM | TGI |
|---------|--------|------------------|-----------|------|-----|
| **CPU** | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ |
| **GPU NVIDIA** | ⭐⭐ | ⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **Memoria RAM** | Media | Baja | Baja | Alta | Alta |
| **Latencia** | Baja | Muy baja | Muy baja | Baja | Baja |
| **Throughput** | Medio | Bajo | Medio | Alto | Alto |
| **Batching** | ❌ | ❌ | ❌ | ✅ | ✅ |
| **Optimización** | General | - | - | PagedAttention | FlashAttention |

**Notas:**
- **Latencia**: Tiempo para 1 request
- **Throughput**: Requests por segundo con carga
- **Batching**: Procesar múltiples requests simultáneamente
- **PagedAttention**: Optimiza uso de memoria GPU (más requests simultáneos)
- **FlashAttention**: Optimiza velocidad de cálculo (inferencia más rápida)

### 6.3 Guía de Decisión: ¿Cuál usar?

## 🎯 Elige tu patrón

### ¿Estás aprendiendo sobre LLMs locales?
→ **Ollama**
- Más fácil de configurar
- Documentación excelente
- Comunidad activa

### ¿Solo necesitas Python sin servidor?
→ **llama-cpp-python**
- Script que corre una vez
- No quieres daemon en background
- Control directo del modelo

### ¿Necesitas máximo rendimiento en CPU?
→ **llama.cpp compilado**
- C++ puro (sin overhead)
- Control total de compilación
- Servidor HTTP ligero

### ¿Tienes GPU NVIDIA y vas a producción?
→ **vLLM o TGI**

| Elige vLLM si... | Elige TGI si... |
|------------------|-----------------|
| Prefieres `pip install` | Prefieres Docker |
| Necesitas PagedAttention | Usas mucho HuggingFace |
| Tu equipo ya lo conoce | Quieres FlashAttention |

### ¿Quieres portabilidad del código?
→ **llama.cpp, vLLM o TGI** (todos usan API OpenAI)
- Mismo código funciona con OpenAI
- Solo cambias `base_url` y `api_key`

### ¿Múltiples usuarios simultáneos?
→ **vLLM o TGI**
- Continuous batching
- Mejor gestión de concurrencia

### 6.4 Diagrama de Decisión

```
¿Tienes GPU NVIDIA?
├─ Sí
│  └─ ¿Múltiples usuarios / alto tráfico?
│     ├─ Sí
│     │  └─ ¿Prefieres Docker?
│     │     ├─ Sí → TGI
│     │     └─ No → vLLM
│     └─ No → Ollama o llama.cpp
│
└─ No (solo CPU)
   └─ ¿Quieres servidor HTTP?
      ├─ Sí
      │  └─ ¿Facilidad o rendimiento?
      │     ├─ Facilidad → Ollama
      │     └─ Rendimiento → llama.cpp compilado
      │
      └─ No
         └─ llama-cpp-python
```

### 6.5 LLMs Locales en Producción: Casos Reales

**¿Realmente las empresas usan modelos locales en producción?**

**Sí, absolutamente.** Hay múltiples razones por las que empresas eligen modelos locales:

### ¿Por qué producción local?

| Razón | Ejemplo |
|-------|---------|
| **Privacidad de datos** | Bancos, hospitales, gobierno (datos sensibles no pueden salir) |
| **Latencia** | Trading, juegos (milisegundos importan) |
| **Costo** | Millones de requests/día → GPT-4 es muy caro |
| **Personalización** | Modelos fine-tuned para dominio específico |
| **Sin dependencia externa** | No depender de API de OpenAI |

### Patrones más usados en Producción

| Patrón | Uso en Producción | Empresas/Casos |
|--------|-------------------|----------------|
| **vLLM** | ⭐⭐⭐ El más usado | Startups de AI, empresas tech |
| **TGI** | ⭐⭐⭐ Muy usado | HuggingFace, empresas grandes |
| **llama.cpp** | ⭐⭐ Común | Dispositivos edge, IoT, móviles |
| **Ollama** | ⭐ Desarrollo/prototipos | Equipos pequeños, MVPs |

### Empresas conocidas usando LLMs locales

| Empresa/Proyecto | Patrón | Uso |
|------------------|--------|-----|
| **Brave** | llama.cpp | Asistente en navegador |
| **Apple** | Modelos propios | Siri, on-device ML |
| **Samsung** | Modelos propios | Galaxy AI |
| **Replit** | vLLM + modelos propios | Code completion |
| **Notion** | vLLM/TGI | AI features |
| **Discord** | vLLM/TGI | Moderación, respuestas |

### Costos aproximados

| Setup | Costo mensual | Capacidad |
|-------|---------------|-----------|
| **1x T4 GPU (16GB)** | ~$150-300/mes | ~10-50 req/s |
| **1x A10G (24GB)** | ~$300-500/mes | ~30-100 req/s |
| **1x A100 (80GB)** | ~$1500-3000/mes | ~100-500 req/s |
| **GPT-4 API (1M tokens)** | ~$30-60 | Variable |

**Breakeven:** Si haces más de ~50,000 requests/día, modelos locales son más económicos.

### 6.6 Código LangChain Lado a Lado

**Ollama:**
```python
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2")
response = llm.invoke("pregunta")
print(response.content)
```

**llama-cpp-python:**
```python
from langchain_community.llms import LlamaCpp
llm = LlamaCpp(model_path="./models/modelo.gguf", n_ctx=2048)
response = llm.invoke("pregunta")
print(response)
```

**llama.cpp compilado:**
```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url="http://localhost:8080/v1", api_key="dummy", model="local")
response = llm.invoke("pregunta")
print(response.content)
```

**vLLM:**
```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url="http://localhost:8000/v1", api_key="dummy", model="TinyLlama/...")
response = llm.invoke("pregunta")
print(response.content)
```

**TGI:**
```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url="http://localhost:8080/v1", api_key="dummy", model="TinyLlama/...")
response = llm.invoke("pregunta")
print(response.content)
```

**Observaciones:**
- Ollama: Código más simple, solo nombre del modelo
- llama-cpp-python: No usa servidor, carga directo
- llama.cpp, vLLM y TGI: Usan misma clase (`ChatOpenAI`), solo cambia puerto
- vLLM y TGI son **intercambiables** (mismo código, diferente puerto)

---
# PARTE 7: Recursos y Próximos Pasos

## 📚 Recursos de Aprendizaje

### Documentación Oficial

**Ollama:**
- Sitio oficial: https://ollama.com
- Documentación: https://github.com/ollama/ollama/blob/main/docs/api.md
- Modelos: https://ollama.com/library
- LangChain: https://python.langchain.com/docs/integrations/llms/ollama

**llama.cpp:**
- GitHub: https://github.com/ggerganov/llama.cpp
- Documentación server: https://github.com/ggerganov/llama.cpp/blob/master/examples/server/README.md

**llama-cpp-python:**
- GitHub: https://github.com/abetlen/llama-cpp-python
- Documentación: https://llama-cpp-python.readthedocs.io/
- LangChain: https://python.langchain.com/docs/integrations/llms/llamacpp

**vLLM:**
- Sitio oficial: https://vllm.ai
- GitHub: https://github.com/vllm-project/vllm
- Documentación: https://docs.vllm.ai/
- Modelos compatibles: https://docs.vllm.ai/en/latest/models/supported_models.html

**TGI (Text Generation Inference):**
- GitHub: https://github.com/huggingface/text-generation-inference
- Documentación: https://huggingface.co/docs/text-generation-inference
- Docker Hub: https://hub.docker.com/r/ghcr.io/huggingface/text-generation-inference
- API Reference: https://huggingface.github.io/text-generation-inference/

### Repositorios de Modelos

**Ollama Library:**
- URL: https://ollama.com/library
- Formato: GGUF cuantizado
- Descarga: `ollama pull <modelo>`

**HuggingFace:**
- URL: https://huggingface.co/models
- Formatos: GGUF, SafeTensors, PyTorch
- Usuario recomendado para GGUF: https://huggingface.co/TheBloke

**Modelos recomendados por tamaño:**

| Tamaño | Modelo | Parámetros | VRAM | Uso |
|--------|--------|------------|------|-----|
| **Tiny** | TinyLlama | 1.1B | 1GB | Pruebas |
| **Small** | Llama 3.2 | 3B | 3GB | General |
| **Medium** | Llama 3.1 | 8B | 6GB | Calidad |
| **Large** | Llama 3.1 | 70B | 40GB | Máxima calidad |

## 🎯 Próximos Pasos

### 1. Profundizar en LangChain
- **Chains**: Encadenar múltiples llamadas al LLM
- **Agents**: LLMs que usan herramientas
- **RAG**: Retrieval Augmented Generation (búsqueda + generación)
- **Memory**: Mantener contexto entre conversaciones

### 2. Optimización de Modelos
- **Cuantización**: Reducir tamaño de modelos (Q4, Q8, etc.)
- **LoRA**: Fine-tuning eficiente
- **GPTQ/AWQ**: Técnicas de compresión

### 3. Producción
- **Docker**: Containerizar tus servidores
- **Nginx**: Proxy reverso para load balancing
- **Monitoring**: Prometheus + Grafana
- **Escalado**: Kubernetes para múltiples instancias

### 4. Aplicaciones Prácticas
- **Chatbots**: Interfaces conversacionales
- **RAG Systems**: Q&A sobre documentos
- **Code Assistants**: Generación de código
- **Data Analysis**: Análisis de datos con LLMs

## 💡 Ejercicios Propuestos

### Ejercicio 1: Comparación de Rendimiento
Mide el tiempo de respuesta de cada patrón para la misma pregunta:
```python
import time

prompt = "Explica qué es una API REST en 3 párrafos"

# Medir Ollama
start = time.time()
response = ollama_llm.invoke(prompt)
ollama_time = time.time() - start

# Repetir para llama-cpp-python, llama.cpp, vLLM
# ¿Cuál es más rápido? ¿Por qué?
```

### Ejercicio 2: Streaming
Implementa streaming (respuesta palabra por palabra) con Ollama:
```python
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")

for chunk in llm.stream("Cuenta una historia corta"):
    print(chunk.content, end="", flush=True)
```
Implementa lo mismo con vLLM y compara.

### Ejercicio 3: Multi-turn Conversation
Crea una conversación con memoria usando LangChain:
```python
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory)

conversation.predict(input="Mi nombre es Juan")
conversation.predict(input="¿Cómo me llamo?")
```

### Ejercicio 4: RAG Simple
Implementa un sistema de búsqueda sobre tus documentos:
1. Cargar documentos
2. Crear embeddings (con sentence-transformers)
3. Almacenar en vector store (FAISS)
4. Hacer preguntas que busquen en los documentos

### Ejercicio 5: API Propia
Crea tu propia API con FastAPI que use uno de los patrones:
```python
from fastapi import FastAPI
from langchain_ollama import ChatOllama

app = FastAPI()
llm = ChatOllama(model="llama3.2")

@app.post("/generate")
def generate(prompt: str):
    response = llm.invoke(prompt)
    return {"response": response.content}
```

## 🎓 Conclusiones

### Lo que aprendiste en este notebook

1. **5 Patrones diferentes** para usar modelos locales
   - Ollama (fácil, automático)
   - llama-cpp-python (Python directo)
   - llama.cpp compilado (control total)
   - vLLM (producción GPU - PagedAttention)
   - TGI (producción GPU - FlashAttention)

2. **3 Métodos de uso** por patrón
   - LangChain (alto nivel)
   - requests (nivel medio)
   - curl (bajo nivel)

3. **Conceptos fundamentales**
   - HTTP, JSON, APIs REST
   - Cliente-servidor, daemon
   - Servidores locales (localhost)
   - Puertos (11434, 8080, 8000)
   - Formatos de modelos (GGUF vs HuggingFace)
   - Cuantización
   - PagedAttention vs FlashAttention

4. **Ecosistema de LLMs locales**
   - Repositorios de modelos
   - Herramientas de inferencia
   - APIs compatibles con OpenAI
   - LangChain como capa de abstracción

### Tarjeta de Referencia Rápida

```
┌─────────────────────────────────────────────────────────────┐
│                    REFERENCIA RÁPIDA                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ OLLAMA                    Puerto: 11434                     │
│ ├─ Instalar: curl -fsSL https://ollama.com/install.sh | sh │
│ ├─ Modelo: ollama pull llama3.2                            │
│ ├─ Servidor: ollama serve                                  │
│ └─ LangChain: ChatOllama(model="llama3.2")                 │
│                                                             │
│ LLAMA-CPP-PYTHON          Sin servidor                      │
│ ├─ Instalar: pip install llama-cpp-python                  │
│ ├─ Modelo: wget ... -O modelo.gguf                         │
│ └─ LangChain: LlamaCpp(model_path="modelo.gguf")           │
│                                                             │
│ LLAMA.CPP COMPILADO       Puerto: 8080                      │
│ ├─ Compilar: make llama-server                             │
│ ├─ Servidor: ./llama-server -m modelo.gguf --port 8080     │
│ └─ LangChain: ChatOpenAI(base_url="localhost:8080/v1")     │
│                                                             │
│ vLLM                      Puerto: 8000                      │
│ ├─ Instalar: pip install vllm                              │
│ ├─ Servidor: vllm serve TinyLlama/... --port 8000          │
│ └─ LangChain: ChatOpenAI(base_url="localhost:8000/v1")     │
│                                                             │
│ TGI                       Puerto: 8080 (Docker)             │
│ ├─ Docker: docker run --gpus all -p 8080:80 ghcr.io/...    │
│ ├─ Modelo: Se especifica con --model-id                    │
│ └─ LangChain: ChatOpenAI(base_url="localhost:8080/v1")     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Mensaje Final

Has completado un recorrido completo por el ecosistema de LLMs locales. Ahora tienes:

✅ Conocimiento de **5 patrones diferentes**  
✅ Capacidad de elegir el mejor para cada caso  
✅ Entendimiento de HTTP, APIs y cliente-servidor  
✅ Conocimiento de optimizaciones GPU (PagedAttention, FlashAttention)  
✅ Experiencia práctica con LangChain  
✅ Base para construir aplicaciones con LLMs locales  

**Siguiente paso recomendado:**
- Elige el patrón que mejor se adapte a tu caso de uso
- Construye un proyecto pequeño (chatbot, RAG, etc.)
- Experimenta con diferentes modelos
- Explora LangChain más a fondo (chains, agents, RAG)

**¡Buena suerte con tus proyectos de LLMs!** 🚀